# Pump Vibration Anomaly Detector — Colab T4 Training

**Before running:** Runtime → Change runtime type → T4 GPU

Checkpoints are saved to Google Drive every epoch so training survives disconnects.

In [ ]:
# Cell 1: Verify GPU and mount Drive
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/pump-anomaly/checkpoints'
print(f'Drive mounted. Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/YOUR_USERNAME/pump-anomaly-detector.git
%cd pump-anomaly-detector
!pip install -q -r requirements.txt

In [ ]:
# Cell 3: Restore checkpoint from Drive (skip on first run if no checkpoint exists)
import shutil, os
os.makedirs('models/saved', exist_ok=True)
drive_ckpt = f'{DRIVE_DIR}/checkpoint_last.pt'
if os.path.exists(drive_ckpt):
    shutil.copy2(drive_ckpt, 'models/saved/checkpoint_last.pt')
    shutil.copy2(f'{DRIVE_DIR}/checkpoint_best.pt', 'models/saved/checkpoint_best.pt')
    print('Checkpoint restored from Drive — training will resume from last epoch')
else:
    print('No previous checkpoint found — starting fresh')

In [ ]:
# Cell 4: Train
# Runs until Colab 4-hour limit or early stopping fires.
# Saves checkpoint to Drive every epoch — safe to interrupt anytime.
!python train.py \
    --epochs 100 \
    --batch-size 128 \
    --accumulation-steps 4 \
    --window-size 512 \
    --latent-dim 32 \
    --patience 15 \
    --drive-dir {DRIVE_DIR} \
    --resume

In [ ]:
# Cell 5: Plot training curves
import json
import matplotlib.pyplot as plt

with open('models/saved/history.json') as f:
    h = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h['train_loss'], label='Train loss')
ax.plot(h['val_loss'], label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Reconstruction loss — pump autoencoder')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
best = min(h['val_loss'])
best_ep = h['val_loss'].index(best) + 1
print(f'Best val loss: {best:.6f} at epoch {best_ep}')

In [ ]:
# Cell 6: Visualise normal vs anomalous reconstruction
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '.')
from models.autoencoder import PumpAutoencoder
from utils.dataset import generate_synthetic_vibration, generate_anomalous_vibration

model = PumpAutoencoder(input_length=512, latent_dim=32)
ckpt = torch.load('models/saved/checkpoint_best.pt', map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval()

def get_window(sig):
    w = sig[:512].astype(np.float32)
    w = (w - w.mean()) / (w.std() + 1e-8)
    return torch.from_numpy(w).unsqueeze(0).unsqueeze(0)

normal_sig = generate_synthetic_vibration(1024)
anomaly_sig = generate_anomalous_vibration(1024, anomaly_type='bearing')

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for i, (sig, label) in enumerate([(normal_sig, 'Normal'), (anomaly_sig, 'Anomaly (bearing)')]):
    x = get_window(sig)
    with torch.no_grad():
        x_hat = model(x)
    orig = x.squeeze().numpy()
    recon = x_hat.squeeze().numpy()
    err = float(((orig - recon) ** 2).mean())
    color = 'crimson' if i else 'steelblue'

    axes[i, 0].plot(orig, lw=0.8, label='Original')
    axes[i, 0].plot(recon, lw=0.8, linestyle='--', label='Reconstructed')
    axes[i, 0].set_title(f'{label} — MSE: {err:.5f}')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    residual = np.abs(orig - recon)
    axes[i, 1].fill_between(range(len(residual)), residual, alpha=0.7, color=color)
    axes[i, 1].set_title(f'{label} — Residual |x - x_hat|')
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reconstruction_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 7: Download ONNX model + threshold for local CPU inference
from google.colab import files
files.download('models/saved/pump_autoencoder.onnx')
files.download('models/saved/threshold.json')
print('Place both files in models/saved/ on your local machine, then run:')
print('  python infer.py --demo')